In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_1.py ──
"""
Shared infrastructure for MLFP03 Exercise 1 — Feature Engineering
and Feature Selection on ICU data.

Contains:
    - ICU multi-table loading with temporal casts
    - Point-in-time feature builders (vitals, medications, labs)
    - ExperimentTracker / ConnectionManager setup
    - Shared prep helpers for feature-selection methods
    - Plotting helpers for feature rankings

Technique-specific code (mutual_info, RFE, Lasso paths, schema
validation) does NOT belong here — it lives in the per-technique files.
"""

import asyncio
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from dotenv import load_dotenv

from kailash.db import ConnectionManager
from kailash_ml import DataExplorer
from kailash_ml.engines.experiment_tracker import ExperimentTracker



# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()
load_dotenv()

OUTPUT_DIR = Path("outputs") / "mlfp03_ex1_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_DB = "sqlite:///mlfp03_experiments.db"
EXPERIMENT_NAME = "mlfp03_healthcare_features"

_DT_FMT = "%Y-%m-%d %H:%M:%S"

VITAL_COLS: list[str] = [
    "heart_rate",
    "systolic_bp",
    "diastolic_bp",
    "temperature",
    "spo2",
    "respiratory_rate",
]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — ICU multi-table
# ════════════════════════════════════════════════════════════════════════


def load_icu_tables() -> dict[str, pl.DataFrame]:
    """Load all five ICU tables and cast timestamp columns to datetime.

    Returns a dict with keys: patients, admissions, vitals (long format),
    medications, labs.

    Vitals are returned in LONG format (columns: admission_id, patient_id,
    timestamp, vital_name, value) — the monolithic exercise unpivots them
    inline, but every technique file wants the long form.
    """
    loader = MLFPDataLoader()
    patients = loader.load("mlfp02", "icu_patients.parquet")
    admissions = loader.load("mlfp02", "icu_admissions.parquet")
    vitals = loader.load("mlfp02", "icu_vitals.parquet")
    medications = loader.load("mlfp02", "icu_medications.parquet")
    labs = loader.load("mlfp02", "icu_labs.parquet")

    # Cast timestamps — polars reads them as strings from the parquet
    admissions = admissions.with_columns(
        pl.col("admit_time").str.to_datetime(_DT_FMT),
        pl.col("discharge_time").str.to_datetime(_DT_FMT),
    )
    medications = medications.with_columns(
        pl.col("start_time").str.to_datetime(_DT_FMT),
        pl.col("end_time").str.to_datetime(_DT_FMT),
    )
    if "timestamp" in labs.columns and labs["timestamp"].dtype == pl.String:
        labs = labs.with_columns(pl.col("timestamp").str.to_datetime(_DT_FMT))

    # Vitals: attach patient_id via admissions join, cast timestamp, unpivot
    vitals = vitals.join(
        admissions.select(["admission_id", "patient_id"]),
        on="admission_id",
        how="left",
    ).with_columns(pl.col("timestamp").str.to_datetime(_DT_FMT))

    present = [c for c in VITAL_COLS if c in vitals.columns]
    if present:
        vitals = vitals.unpivot(
            present,
            index=["admission_id", "patient_id", "timestamp"],
            variable_name="vital_name",
            value_name="value",
        )

    return {
        "patients": patients,
        "admissions": admissions,
        "vitals": vitals,
        "medications": medications,
        "labs": labs,
    }


# ════════════════════════════════════════════════════════════════════════
# FEATURE BUILDERS — point-in-time aggregates
# ════════════════════════════════════════════════════════════════════════


def build_vital_features(
    vitals: pl.DataFrame, admissions: pl.DataFrame
) -> pl.DataFrame:
    """Aggregate long-format vitals per admission with temporal correctness.

    Only uses vital readings recorded BETWEEN admit_time and discharge_time
    for each admission. Returns one row per admission with columns:
        {vital}_{mean,std,min,max,range,trend,count,cv}
    """
    # Vitals already carries admission_id and patient_id. Join only to pull
    # in the admit/discharge window; drop admissions' patient_id on the
    # way in to avoid duplicate columns.
    filtered = vitals.join(
        admissions.select("admission_id", "admit_time", "discharge_time"),
        on="admission_id",
        how="inner",
    ).filter(
        (pl.col("timestamp") >= pl.col("admit_time"))
        & (pl.col("timestamp") <= pl.col("discharge_time"))
    )

    names = filtered["vital_name"].unique().to_list()
    aggs: list[pl.DataFrame] = []
    for vital in names:
        agg = (
            filtered.filter(pl.col("vital_name") == vital)
            .group_by("admission_id")
            .agg(
                pl.col("value").mean().alias(f"{vital}_mean"),
                pl.col("value").std().alias(f"{vital}_std"),
                pl.col("value").min().alias(f"{vital}_min"),
                pl.col("value").max().alias(f"{vital}_max"),
                (pl.col("value").max() - pl.col("value").min()).alias(f"{vital}_range"),
                (pl.col("value").last() - pl.col("value").first()).alias(
                    f"{vital}_trend"
                ),
                pl.col("value").count().alias(f"{vital}_count"),
                (pl.col("value").std() / pl.col("value").mean()).alias(f"{vital}_cv"),
            )
        )
        aggs.append(agg)

    # Merge vital aggregates via full-outer join with coalesced key.
    out = aggs[0]
    for a in aggs[1:]:
        out = out.join(a, on="admission_id", how="full", coalesce=True)
    return out


def build_medication_features(
    medications: pl.DataFrame, admissions: pl.DataFrame
) -> pl.DataFrame:
    """Flag high-risk medications and count distinct drugs per admission."""
    return (
        medications.join(
            admissions.select(
                "patient_id", "admission_id", "admit_time", "discharge_time"
            ),
            on="admission_id",
            how="inner",
        )
        .filter(
            (pl.col("start_time") >= pl.col("admit_time"))
            & (pl.col("start_time") <= pl.col("discharge_time"))
        )
        .group_by("admission_id")
        .agg(
            pl.col("drug_name").n_unique().alias("n_unique_medications"),
            pl.col("drug_name").count().alias("n_medication_doses"),
            pl.col("drug_name")
            .str.contains("(?i)vasopressor|norepinephrine|dopamine")
            .any()
            .alias("received_vasopressors"),
            pl.col("drug_name")
            .str.contains("(?i)antibiotic|vancomycin|meropenem")
            .any()
            .alias("received_antibiotics"),
            pl.col("drug_name")
            .str.contains("(?i)propofol|midazolam|fentanyl")
            .any()
            .alias("received_sedation"),
        )
    )


def build_lab_features(labs: pl.DataFrame, admissions: pl.DataFrame) -> pl.DataFrame:
    """Aggregate lab results per admission with abnormal-flag counts."""
    return (
        labs.join(
            admissions.select("admission_id", "admit_time", "discharge_time"),
            on="admission_id",
            how="inner",
        )
        .filter(
            (pl.col("timestamp") >= pl.col("admit_time"))
            & (pl.col("timestamp") <= pl.col("discharge_time"))
        )
        .group_by("admission_id")
        .agg(
            pl.col("test_name").n_unique().alias("n_unique_labs"),
            pl.col("value").count().alias("n_lab_results"),
            (pl.col("flag") != "normal").sum().alias("n_abnormal_labs"),
            pl.col("value").mean().alias("lab_value_mean"),
            pl.col("value").std().alias("lab_value_std"),
        )
    )


def build_full_feature_frame(tables: dict[str, pl.DataFrame]) -> pl.DataFrame:
    """End-to-end feature matrix used by every technique file.

    Composes patients + admissions with vital / medication / lab / derived /
    interaction features. Every technique file calls this so the starting
    feature matrix is identical — only the SELECTION method differs.
    """
    patients = tables["patients"]
    admissions = tables["admissions"]

    patient_admissions = patients.join(admissions, on="patient_id", how="inner")

    features = patient_admissions.clone()
    vf = build_vital_features(tables["vitals"], admissions)
    features = features.join(vf, on="admission_id", how="left")

    # Vital stat columns (std, trend, cv, ...) are legitimately null
    # when a patient has only 1 reading for a given vital, or when a
    # vital was never sampled. Fill with 0 so downstream selection
    # methods (sklearn) see no nulls.
    vital_stat_cols = [
        c
        for c in features.columns
        if any(
            c.endswith(f"_{s}")
            for s in ("mean", "std", "min", "max", "range", "trend", "count", "cv")
        )
    ]
    features = features.with_columns(
        *[pl.col(c).fill_null(0.0) for c in vital_stat_cols]
    )

    mf = build_medication_features(tables["medications"], admissions)
    features = features.join(mf, on="admission_id", how="left")

    lf = build_lab_features(tables["labs"], admissions)
    features = features.join(lf, on="admission_id", how="left")

    # Derived features
    features = features.with_columns(
        (pl.col("n_abnormal_labs") / pl.col("n_lab_results").clip(lower_bound=1)).alias(
            "abnormal_lab_ratio"
        ),
        (pl.col("n_medication_doses") / pl.col("los_days").clip(lower_bound=1)).alias(
            "medication_intensity"
        ),
        (pl.col("n_lab_results") / pl.col("los_days").clip(lower_bound=1)).alias(
            "lab_intensity"
        ),
        (pl.col("n_unique_medications") > 10).alias("polypharmacy_flag"),
    )

    # Null fills for patients without meds / labs
    fill_int = [
        "n_unique_medications",
        "n_medication_doses",
        "n_unique_labs",
        "n_lab_results",
        "n_abnormal_labs",
    ]
    fill_bool = [
        "received_vasopressors",
        "received_antibiotics",
        "received_sedation",
        "polypharmacy_flag",
    ]
    fill_float = [
        "abnormal_lab_ratio",
        "medication_intensity",
        "lab_intensity",
        "lab_value_mean",
        "lab_value_std",
    ]
    features = features.with_columns(
        *[pl.col(c).fill_null(0) for c in fill_int if c in features.columns],
        *[pl.col(c).fill_null(False) for c in fill_bool if c in features.columns],
        *[pl.col(c).fill_null(0.0) for c in fill_float if c in features.columns],
    )

    # Interactions (clinical domain knowledge)
    cols = features.columns
    exprs: list[pl.Expr] = []
    if "heart_rate_mean" in cols and "systolic_bp_mean" in cols:
        exprs.append(
            (
                pl.col("heart_rate_mean")
                / pl.col("systolic_bp_mean").clip(lower_bound=1)
            ).alias("shock_index")
        )
    if "systolic_bp_mean" in cols and "diastolic_bp_mean" in cols:
        exprs.append(
            ((pl.col("systolic_bp_mean") + 2 * pl.col("diastolic_bp_mean")) / 3).alias(
                "map_mean"
            )
        )
    if "temperature_mean" in cols and "heart_rate_mean" in cols:
        exprs.append(
            (pl.col("temperature_mean") * pl.col("heart_rate_mean")).alias(
                "fever_tachycardia"
            )
        )
    exprs.append(
        (pl.col("medication_intensity") * pl.col("abnormal_lab_ratio")).alias(
            "treatment_burden_score"
        )
    )
    features = features.with_columns(*exprs)

    # Fill any nulls introduced by the interactions
    for name in (
        "shock_index",
        "map_mean",
        "fever_tachycardia",
        "treatment_burden_score",
    ):
        if name in features.columns:
            features = features.with_columns(pl.col(name).fill_null(0.0))

    return features


# ════════════════════════════════════════════════════════════════════════
# SELECTION INPUT PREP
# ════════════════════════════════════════════════════════════════════════


def prepare_selection_inputs(
    features: pl.DataFrame,
) -> tuple[list[str], np.ndarray, np.ndarray]:
    """Return (feature_cols, X, y_binary) for every feature-selection method.

    - Drops ID columns and the target from the feature matrix
    - Coerces bool/int/float columns only (selection methods require numeric)
    - Replaces NaN / inf with bounded numbers
    - Builds a binary target: mortality if present, otherwise los_days > median
    """
    id_cols = {"patient_id", "admission_id", "admit_time", "discharge_time"}
    target_col = "mortality" if "mortality" in features.columns else "los_days"
    exclude = id_cols | {target_col}

    numeric_dtypes = {pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.Boolean}
    feature_cols = [
        c
        for c in features.columns
        if c not in exclude and features[c].dtype in numeric_dtypes
    ]

    X = features.select(feature_cols).to_numpy().astype(np.float64)
    X = np.nan_to_num(X, nan=0.0, posinf=1e6, neginf=-1e6)

    if target_col == "mortality":
        y = features["mortality"].to_numpy().astype(np.float64).ravel()
    else:
        median_los = features["los_days"].median()
        y = (
            (features["los_days"] > median_los)
            .cast(pl.Int32)
            .to_numpy()
            .ravel()
            .astype(np.float64)
        )
    y_binary = (
        (y > np.median(y)).astype(int) if target_col != "mortality" else y.astype(int)
    )
    return feature_cols, X, y_binary


# ════════════════════════════════════════════════════════════════════════
# EXPERIMENT TRACKING
# ════════════════════════════════════════════════════════════════════════


async def setup_tracking() -> tuple[ConnectionManager, ExperimentTracker, str]:
    """Initialize ConnectionManager + ExperimentTracker and create the M3
    experiment. Every technique file in ex_1 logs into the same experiment
    so selection runs are directly comparable.
    """
    conn = ConnectionManager(EXPERIMENT_DB)
    await conn.initialize()
    tracker = ExperimentTracker(conn)
    experiment_id = await tracker.create_experiment(
        name=EXPERIMENT_NAME,
        description="Feature engineering experiments on ICU data — Module 3",
        tags=["mlfp03", "healthcare", "feature-engineering"],
    )
    return conn, tracker, experiment_id


def setup_tracking_sync() -> tuple[ConnectionManager, ExperimentTracker, str]:
    """Sync wrapper for setup_tracking — convenience for non-async files."""
    return asyncio.run(setup_tracking())


async def log_selection_run(
    tracker: ExperimentTracker,
    experiment_id: str,
    *,
    run_name: str,
    method: str,
    selected_features: list[str],
    total_features: int,
    extra_params: dict[str, str] | None = None,
    extra_metrics: dict[str, float] | None = None,
) -> str:
    """Log a feature-selection run to ExperimentTracker. Returns run id."""
    params = {
        "method": method,
        "n_features_total": str(total_features),
        "n_features_selected": str(len(selected_features)),
        "selected_features": ",".join(selected_features[:30]),
    }
    if extra_params:
        params.update(extra_params)
    metrics = {
        "n_features_selected": float(len(selected_features)),
        "selection_ratio": float(len(selected_features)) / max(1, total_features),
    }
    if extra_metrics:
        metrics.update(extra_metrics)

    async with tracker.run(experiment_id, run_name=run_name) as run:
        await run.log_params(params)
        await run.log_metrics(metrics)
        await run.set_tag("domain", "clinical")
        await run.set_tag("selection_family", method)
        run_id = getattr(run, "id", run_name)
    return run_id


# ════════════════════════════════════════════════════════════════════════
# REPORTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def print_ranking(
    title: str, ranking: list[tuple[str, float]], *, top: int = 15
) -> None:
    """Print a ranked feature list with a simple ASCII bar chart."""
    print(f"\n=== {title} ===")
    print(f"{'Feature':<35} {'Score':>10}")
    print("-" * 48)
    if not ranking:
        print("  (empty ranking)")
        return
    max_score = max(abs(s) for _, s in ranking[:top]) or 1.0
    for name, score in ranking[:top]:
        bar_len = int(abs(score) / max_score * 20)
        bar = "#" * bar_len
        print(f"  {name:<33} {score:>10.4f}  {bar}")


def save_ranking_csv(
    ranking: list[tuple[str, float]], filename: str, score_col: str = "score"
) -> Path:
    """Persist a ranking as CSV into OUTPUT_DIR. Returns the file path."""
    path = OUTPUT_DIR / filename
    pl.DataFrame(
        {"feature": [n for n, _ in ranking], score_col: [s for _, s in ranking]}
    ).write_csv(path)
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 1.1: Clinical Feature Engineering with
#                         Point-in-Time Correctness
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Join five ICU tables with temporal correctness (no future leakage)
#   - Aggregate irregular time-series vitals into per-admission statistics
#   - Flag clinically meaningful medication and lab patterns
#   - Engineer interaction features that encode domain knowledge (shock
#     index, mean arterial pressure, fever-tachycardia product)
#   - Apply to early-warning scoring at Singapore General Hospital
#
# PREREQUISITES: MLFP02 complete (polars group-by, joins, temporal filters)
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — why point-in-time correctness matters
#   2. Build — load tables, aggregate vitals, meds, labs
#   3. Train — there is no training; we BUILD the full feature matrix
#   4. Visualise — preview the engineered columns + interaction distributions
#   5. Apply — Singapore General Hospital early-warning scoring (S$ impact)
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import plotly.express as px
import plotly.graph_objects as go
import polars as pl



## THEORY — Why Point-in-Time Correctness Matters


In [ ]:
# A feature built from "all vitals this patient ever had" leaks the
# future. If prediction time is the moment of ICU admission, then the
# patient's discharge-day vitals do not exist yet — using them inflates
# validation accuracy and fails catastrophically in production.
#
# The fix is a temporal filter: every feature only uses data recorded
# BETWEEN admit_time and discharge_time for THAT admission. The same
# rule applies to medications (start_time), labs (timestamp), and any
# derived feature.
#
# Analogy: imagine building a stock-price model that accidentally uses
# tomorrow's close as today's feature. Your backtest looks amazing; your
# live trading loses every penny. Medical ML has the same failure mode,
# but the stakes are higher — a biased model kills patients, not pennies.



## TASK 2 — BUILD: load tables and construct per-admission features


In [ ]:
print("\n" + "=" * 70)
print("  Clinical Feature Engineering — ICU Multi-Table")
print("=" * 70)

tables = load_icu_tables()
for name, df in tables.items():
    print(f"  {name}: {df.shape}")

# The shared helper encodes the full feature contract: vital aggregates
# per admission (mean/std/min/max/range/trend/count/cv), medication flags
# (vasopressors, antibiotics, sedation), lab ratios, and clinical
# interaction features. Every technique file in this exercise starts from
# the same matrix — only the SELECTION method differs.
features = build_full_feature_frame(tables)

print(f"\nFeature matrix: {features.shape}")
print(f"Total columns: {len(features.columns)}")



### Checkpoint 1


In [ ]:
assert features.height > 0, "Task 2: feature matrix must not be empty"
assert "shock_index" in features.columns, "Task 2: shock_index (interaction) missing"
assert "abnormal_lab_ratio" in features.columns, "Task 2: lab ratio missing"
assert features["abnormal_lab_ratio"].null_count() == 0, "Task 2: null in lab ratio"
print("\n[ok] Checkpoint 1 passed — feature matrix built\n")

# INTERPRETATION: The _count suffix columns are particularly valuable —
# a patient with heart_rate_count = 120 in a 24h stay (5/hour) is being
# monitored far more intensively than one with count = 8. The coefficient
# of variation (_cv) captures NORMALISED volatility: a heart rate that
# oscillates wildly has high CV regardless of baseline.



## TASK 3 — TRAIN (no training — quality audit of the feature matrix)


In [ ]:
# Feature engineering has no gradient-descent loop. The "training" step
# is a deterministic construction, and we validate it by auditing the
# null rate, dtype coverage, and clinical interaction distributions.

print("\n--- Feature Matrix Quality Audit ---")
# Null rate across NUMERIC + BOOLEAN columns only. Stat columns like
# {vital}_std legitimately produce nulls when a patient has a single
# reading, and full-outer vital joins leave nulls for patients who were
# never sampled for a given vital — both are expected, not defects.
null_rate = sum(features[c].null_count() for c in features.columns) / (
    features.height * len(features.columns)
)
numeric_cols = [
    c
    for c in features.columns
    if features[c].dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
]
bool_cols = [c for c in features.columns if features[c].dtype == pl.Boolean]

print(f"  Rows:        {features.height}")
print(f"  Columns:     {len(features.columns)}")
print(f"  Numeric:     {len(numeric_cols)}")
print(f"  Boolean:     {len(bool_cols)}")
print(f"  Global null rate: {null_rate:.4f}")



### Checkpoint 2


In [ ]:
assert null_rate < 0.20, (
    f"Task 3: null rate {null_rate:.4f} exceeds 20%. A high null rate "
    "means the temporal filter dropped too many rows — investigate the "
    "admit_time / discharge_time coverage before moving on."
)
print("\n[ok] Checkpoint 2 passed — feature quality audit OK\n")



## TASK 4 — VISUALISE the interaction features


In [ ]:
# Visual proof: the interaction features should show clinically plausible
# distributions. Shock index > 0.9 is a known emergency marker; MAP
# should centre around 70-90 mmHg for most patients.

print("\n--- Clinical Interaction Feature Distributions ---")
interaction_cols = [
    c
    for c in ("shock_index", "map_mean", "fever_tachycardia", "treatment_burden_score")
    if c in features.columns
]
summary = features.select(
    *[pl.col(c).mean().alias(f"{c}_mean") for c in interaction_cols],
    *[pl.col(c).std().alias(f"{c}_std") for c in interaction_cols],
)
for c in interaction_cols:
    mean = summary[f"{c}_mean"].item() or 0.0
    std = summary[f"{c}_std"].item() or 0.0
    print(f"  {c:<25} mean={mean:>10.3f}  std={std:>10.3f}")

# Rough histogram of shock_index (clinically actionable ranges)
if "shock_index" in features.columns:
    print("\n  shock_index buckets (clinical interpretation):")
    buckets = (
        features.select(
            pl.when(pl.col("shock_index") < 0.7)
            .then(pl.lit("normal (<0.7)"))
            .when(pl.col("shock_index") < 0.9)
            .then(pl.lit("concerning (0.7-0.9)"))
            .when(pl.col("shock_index") < 1.2)
            .then(pl.lit("emergency (0.9-1.2)"))
            .otherwise(pl.lit("critical (>=1.2)"))
            .alias("bucket")
        )
        .group_by("bucket")
        .agg(pl.len().alias("n"))
        .sort("n", descending=True)
    )
    for row in buckets.iter_rows(named=True):
        bar = "#" * min(40, int(row["n"] / 5))
        print(f"    {row['bucket']:<24} {row['n']:>5}  {bar}")

# --- Correlation heatmap of engineered features ---
corr_cols = [c for c in interaction_cols if c in features.columns]
corr_cols += [
    c for c in features.columns if c.endswith("_mean") and c not in corr_cols
][:8]
corr_df = features.select([pl.col(c).cast(pl.Float64) for c in corr_cols]).to_pandas()
corr_matrix = corr_df.corr()
fig_heat = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation Heatmap — Engineered Clinical Features",
)
fig_heat.update_layout(width=800, height=700)
heat_path = OUTPUT_DIR / "ex1_01_correlation_heatmap.html"
fig_heat.write_html(str(heat_path))
print(f"\n  Saved: {heat_path}")

# --- Feature distribution histograms ---
hist_cols = [c for c in interaction_cols if c in features.columns][:4]
fig_hist = go.Figure()
for col in hist_cols:
    vals = features[col].drop_nulls().to_list()
    fig_hist.add_trace(go.Histogram(x=vals, name=col, opacity=0.6, nbinsx=40))
fig_hist.update_layout(
    title="Distribution of Clinical Interaction Features",
    xaxis_title="Value",
    yaxis_title="Count",
    barmode="overlay",
    height=450,
)
hist_path = OUTPUT_DIR / "ex1_01_feature_distributions.html"
fig_hist.write_html(str(hist_path))
print(f"  Saved: {hist_path}")



### Checkpoint 3


In [ ]:
assert "shock_index" in features.columns, "Task 4: shock_index missing"
assert (
    features["shock_index"].null_count() == 0
), "Task 4: null in shock_index — check input vital columns"
print("\n[ok] Checkpoint 3 passed — interaction distributions plausible\n")



## TASK 5 — APPLY: Singapore General Hospital Early-Warning Scoring


In [ ]:
# SCENARIO: Singapore General Hospital (SGH) runs ~2,500 ICU admissions
# per year. The clinical informatics team wants an early-warning score
# that flags deteriorating patients 4-6 hours before a code-blue event.
# The current system uses raw vitals (heart rate > 120 = alert); it
# fires hundreds of false alarms per day and the nurses have started
# ignoring the pager.
#
# Why the engineered features solve it:
#   - shock_index (HR / SBP) is a validated early-warning marker that
#     beats either vital alone — it fires ~30% fewer false alarms at
#     the same sensitivity because it captures compensated shock
#   - abnormal_lab_ratio integrates the "everything is drifting off
#     baseline" signal into a single number
#   - medication_intensity and n_unique_medications act as a proxy for
#     clinician concern — more drugs means the team has already
#     escalated, which is itself a predictor
#
# BUSINESS IMPACT: SGH estimates each prevented code-blue saves roughly
# S$18,000 in ICU escalation costs and adds 2.3 disability-adjusted life
# years per patient. If the engineered features reduce false alarms 30%
# and catch even 5 additional deteriorations per month, that is:
#     5 events/month x 12 months x S$18,000 = S$1.08M/year in direct
#     cost avoidance, plus ~140 DALYs preserved. Feature engineering
#     cost: one clinical informatics hire + one data engineer =
#     ~S$350K/year. 3x ROI in year one; 6-8x once the model compounds
#     across more wards.
#
# LIMITATIONS:
#   - Vitals coverage varies by ward — general wards have sparser vital
#     streams than the ICU, so heart_rate_count is confounded by ward
#   - The model still needs calibration per patient cohort (cardiac,
#     trauma, surgical) before it can ship to other hospitals
#   - Leakage auditing (see 05_validation_and_tracking.py) MUST run
#     every time the feature list changes; one leaky feature silently
#     kills the model in production



## REFLECTION


[x] Joined five ICU tables (patients, admissions, vitals, meds, labs)
  [x] Applied point-in-time filters so features cannot leak the future
  [x] Aggregated irregular vital-sign time series into per-admission stats
  [x] Flagged clinically meaningful drug classes via regex
  [x] Computed clinical interaction features from domain knowledge
  [x] Quantified business impact at Singapore General Hospital

  KEY INSIGHT: Domain knowledge dominates algorithmic complexity. The
  shock_index feature is one division, but it encodes decades of
  emergency-medicine research. No deep-learning architecture recovers
  that signal automatically from raw vitals without supervision.

  Next: 02_filter_selection.py — rank all engineered features using
  mutual information and chi-squared, and see which clinical features
  a model-free filter method can recover.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

